In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Silver Order Items Validation").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
bronze_order_items = spark.read.parquet("../data/bronze/order_items")
silver_order_items = spark.read.parquet("../data/silver/order_items")

In [3]:
from pyspark.sql.functions import col, count,when 

In [4]:
print("Bronze Order Items:", bronze_order_items.count())
print("Silver Order Items:", silver_order_items.count())

Bronze Order Items: 112650
Silver Order Items: 112650


In [5]:
silver_order_items.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in silver_order_items.columns
]).show(vertical=True)

-RECORD 0------------------
 order_id            | 0   
 order_item_id       | 0   
 product_id          | 0   
 seller_id           | 0   
 shipping_limit_date | 0   
 price               | 0   
 freight_value       | 0   



In [6]:
silver_order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [7]:
duplicate_silver_order_items =( silver_order_items.groupBy("order_id","order_item_id")
.count()
.filter(col("count")>1)
)
duplicate_silver_order_items.show()


+--------+-------------+-----+
|order_id|order_item_id|count|
+--------+-------------+-----+
+--------+-------------+-----+

